# 02 - Geracao de Imagens Sinteticas com StyleGAN2

Este notebook foi ajustado para **usar um modelo StyleGAN2-ADA ja treinado**, em vez de iniciar um treino novo.

Objetivo desta etapa:

1. Selecionar uma amostra reprodutivel de **20%** da base `img_align_celeba`.
2. Definir a meta minima de imagens sinteticas em **25%** dessa amostra.
3. Gerar as imagens com um checkpoint pre-treinado compativel com faces alinhadas.


## Referencia usada

Fluxo baseado no repositorio oficial da NVIDIA `stylegan2-ada-pytorch`, usando `generate.py` com um checkpoint pre-treinado.

Observacao importante: o repositorio oficial disponibiliza um checkpoint `CelebA-HQ 256`, que e o modelo oficial mais proximo do dominio da base `CelebA Align` (rostos alinhados).


In [ ]:
from math import ceil
import random
from pathlib import Path
import importlib.util
import sys

from PIL import Image
import matplotlib.pyplot as plt


In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent
DATASET_ROOT = Path('/home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/raw/img_align_celeba')
STYLEGAN_REPO_DIR = PROJECT_ROOT / 'external' / 'stylegan2-ada-pytorch'
STYLEGAN_SYNTHETIC_DIR = PROJECT_ROOT / 'data' / 'processed' / 'stylegan2_synthetic_256'
STYLEGAN_GRID_DIR = PROJECT_ROOT / 'reports' / 'figures' / 'stylegan2_samples'
PRETRAINED_DIR = PROJECT_ROOT / 'artifacts' / 'pretrained'
LOCAL_PRETRAINED_NETWORK = PRETRAINED_DIR / 'stylegan2-celebahq-256x256.pkl'
PRETRAINED_NETWORK_URL = 'https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/transfer-learning-source-nets/celebahq-res256-mirror-paper256-kimg100000-ada-target0.5.pkl'
PRETRAINED_MODEL_NAME = 'StyleGAN2-ADA CelebA-HQ 256'
TRUNCATION_PSI = 0.7
SEED_START = 0
CHUNK_SIZE = 200
SAMPLE_FRACTION = 0.15
SAMPLE_RANDOM_SEED = 42
MAX_REAL_IMAGES = None

for folder in [STYLEGAN_SYNTHETIC_DIR, STYLEGAN_GRID_DIR, PRETRAINED_DIR, PROJECT_ROOT / 'external']:
    folder.mkdir(parents=True, exist_ok=True)

all_real_images = sorted(DATASET_ROOT.glob('*.jpg'))
sample_size = ceil(len(all_real_images) * SAMPLE_FRACTION)
sample_rng = random.Random(SAMPLE_RANDOM_SEED)
sampled_real_images = sorted(sample_rng.sample(all_real_images, sample_size)) if all_real_images else []
if MAX_REAL_IMAGES is not None:
    sampled_real_images = sampled_real_images[:MAX_REAL_IMAGES]

real_image_count = len(sampled_real_images)
synthetic_target_count = ceil(real_image_count * 0.8)
existing_synthetic_images = sorted(STYLEGAN_SYNTHETIC_DIR.glob('*.png'))
remaining_synthetic_count = max(0, synthetic_target_count - len(existing_synthetic_images))
generation_seed_start = SEED_START + len(existing_synthetic_images)

print('Modelo selecionado:', PRETRAINED_MODEL_NAME)
print('Imagens reais encontradas:', len(all_real_images))
print('Fracao da amostra:', SAMPLE_FRACTION)
print('Sample random seed:', SAMPLE_RANDOM_SEED)
print('Imagens reais consideradas na amostra:', real_image_count)
print('Meta minima de imagens sinteticas (80% da amostra):', synthetic_target_count)
print('Imagens sinteticas ja existentes:', len(existing_synthetic_images))
print('Imagens que ainda faltam gerar:', remaining_synthetic_count)


## 1. Garantir o repositorio do StyleGAN2-ADA

Este notebook nao usa comandos de shell. Por isso, o repositorio oficial precisa ja estar disponivel em `external/stylegan2-ada-pytorch`.


In [ ]:
if not STYLEGAN_REPO_DIR.exists():
    raise FileNotFoundError(
        'Repositorio ausente em external/stylegan2-ada-pytorch. '
        'Baixe ou copie o repositorio antes de executar este notebook.'
    )

if str(STYLEGAN_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(STYLEGAN_REPO_DIR))

generate_spec = importlib.util.spec_from_file_location(
    'stylegan2_generate_module',
    STYLEGAN_REPO_DIR / 'generate.py',
)
stylegan_generate = importlib.util.module_from_spec(generate_spec)
assert generate_spec.loader is not None
generate_spec.loader.exec_module(stylegan_generate)

print('Repositorio ja encontrado em:', STYLEGAN_REPO_DIR)
print('Modulo de geracao carregado com sucesso.')


## 2. Resolver o checkpoint pre-treinado

Se existir um arquivo local em `artifacts/pretrained/stylegan2-celebahq-256x256.pkl`, ele tera prioridade. Caso contrario, o notebook usa diretamente a URL oficial da NVIDIA.


In [ ]:
if LOCAL_PRETRAINED_NETWORK.exists():
    pretrained_network = str(LOCAL_PRETRAINED_NETWORK)
    pretrained_network_source = 'arquivo local'
else:
    pretrained_network = PRETRAINED_NETWORK_URL
    pretrained_network_source = 'URL oficial da NVIDIA'

print('Checkpoint selecionado:', pretrained_network)
print('Origem do checkpoint:', pretrained_network_source)


## 3. Definir os blocos de seeds

A geracao e feita em blocos para evitar uma chamada unica muito grande ao `generate.py`.


In [ ]:
def generate_seed_ranges(start, total, chunk_size=200):
    if total <= 0:
        return []

    end = start + total - 1
    current = start
    ranges = []
    while current <= end:
        chunk_end = min(current + chunk_size - 1, end)
        ranges.append(f'{current}-{chunk_end}')
        current = chunk_end + 1
    return ranges

seed_ranges = generate_seed_ranges(generation_seed_start, remaining_synthetic_count, chunk_size=CHUNK_SIZE)
seed_ranges[:5], len(seed_ranges)


## 4. Gerar imagens sinteticas com o modelo pre-treinado

Se a pasta ja tiver imagens suficientes para bater a meta de 25% da amostra, a etapa abaixo nao gera arquivos novos.


In [ ]:
if remaining_synthetic_count == 0:
    print('Geracao pulada: a meta minima ja foi atingida com os arquivos existentes.')
else:
    for seed_range in seed_ranges:
        generate_args = [
            f'--outdir={STYLEGAN_SYNTHETIC_DIR}',
            f'--trunc={TRUNCATION_PSI}',
            f'--seeds={seed_range}',
            f'--network={pretrained_network}',
        ]
        print('generate.py ' + ' '.join(generate_args))
        stylegan_generate.generate_images.main(args=generate_args, standalone_mode=False)


## 5. Inspecionar o resultado


In [ ]:
synthetic_images = sorted(STYLEGAN_SYNTHETIC_DIR.glob('*.png'))
print('Imagens sinteticas geradas:', len(synthetic_images))
print('Meta minima esperada para a amostra:', synthetic_target_count)
print('Diretorio de saida:', STYLEGAN_SYNTHETIC_DIR)
print('Checkpoint utilizado:', pretrained_network)
print('Sample random seed utilizado:', SAMPLE_RANDOM_SEED)


In [ ]:
preview_paths = synthetic_images[:16]
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, image_path in zip(axes.flatten(), preview_paths):
    ax.imshow(Image.open(image_path))
    ax.set_title(image_path.name, fontsize=8)
    ax.axis('off')
for ax in axes.flatten()[len(preview_paths):]:
    ax.axis('off')
plt.tight_layout()
plt.show()


As imagens sinteticas geradas por StyleGAN2 ficam em `data/processed/stylegan2_synthetic_256/`.

Observacao: este notebook nao treina mais uma rede nova sobre `img_align_celeba`. Ele usa um checkpoint pre-treinado `CelebA-HQ 256`, que e o modelo oficial mais proximo para o dominio de rostos alinhados dentro do fluxo do StyleGAN2-ADA. A meta de geracao e calculada sobre uma amostra reprodutivel de 20% da base, controlada por `SAMPLE_RANDOM_SEED`.
